# Hyperparameter Tuning (GCP)
Cross-validate RF and XGBoost on train split, evaluate on val/test.

In [ ]:
import numpy as np
from datetime import datetime
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from xgboost.spark import SparkXGBRegressor

BASE_HDFS = "/user/tiennd"
FEATURE_PATH = f"{BASE_HDFS}/feature_engineering/demand_prediction_features_30m"
OUT_BASE = f"{BASE_HDFS}/results/tuning"
TARGET_COL = "pickup_demand_t1"

feature_cols = [
    "hour", "dow", "month", "is_weekend",
    "lag_6", "lag_12", "lag_336",
    "roll_mean_12", "roll_mean_48", "roll_std_48", "cluster_id",
]

spark = (
    SparkSession.builder
    .appName("HyperparameterTuning_GCP")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", f"hdfs://{BASE_HDFS}/spark-logs")
    .config("spark.executor.instances", "3")
    .config("spark.executor.cores", "3")
    .config("spark.executor.memory", "6g")
    .config("spark.executor.memoryOverhead", "1g")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.memoryOverhead", "1g")
    .config("spark.sql.shuffle.partitions", "96")
    .getOrCreate()
 )
spark.sparkContext.setLogLevel("WARN")

df_all = spark.read.parquet(FEATURE_PATH)
train_df = df_all.filter(F.col("split") == "train").cache()
val_df = df_all.filter(F.col("split") == "val").cache()
test_df = df_all.filter(F.col("split") == "test").cache()

In [ ]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
rmse_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")

def eval_rmse(pred_df):
    pred_df = pred_df.withColumn("prediction", F.when(F.col("prediction") < 0, 0.0).otherwise(F.col("prediction")))
    return float(rmse_eval.evaluate(pred_df))

# RF Grid
rf = RandomForestRegressor(featuresCol="features", labelCol=TARGET_COL, predictionCol="prediction", seed=42)
rf_pipe = Pipeline(stages=[assembler, rf])
rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [30, 60])
    .addGrid(rf.maxDepth, [6, 10])
    .build()
 )

cv_rf = CrossValidator(estimator=rf_pipe, estimatorParamMaps=rf_grid, evaluator=rmse_eval, numFolds=3, parallelism=2)
rf_model = cv_rf.fit(train_df)
rf_val_rmse = eval_rmse(rf_model.transform(val_df))
rf_test_rmse = eval_rmse(rf_model.transform(test_df))
print("RF val RMSE:", rf_val_rmse, "test RMSE:", rf_test_rmse)

# XGBoost Grid (small)
xgb = SparkXGBRegressor(
    features_col="features",
    label_col=TARGET_COL,
    prediction_col="prediction",
    num_workers=3,
    objective="reg:squarederror",
)
xgb_pipe = Pipeline(stages=[assembler, xgb])
xgb_grid = (
    ParamGridBuilder()
    .addGrid(xgb.max_depth, [4, 6])
    .addGrid(xgb.learning_rate, [0.05, 0.1])
    .addGrid(xgb.n_estimators, [50, 100])
    .build()
 )
cv_xgb = CrossValidator(estimator=xgb_pipe, estimatorParamMaps=xgb_grid, evaluator=rmse_eval, numFolds=3, parallelism=2)
xgb_model = cv_xgb.fit(train_df)
xgb_val_rmse = eval_rmse(xgb_model.transform(val_df))
xgb_test_rmse = eval_rmse(xgb_model.transform(test_df))
print("XGB val RMSE:", xgb_val_rmse, "test RMSE:", xgb_test_rmse)

run_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
summary = [
    {"model": "rf", "val_rmse": rf_val_rmse, "test_rmse": rf_test_rmse},
    {"model": "xgb", "val_rmse": xgb_val_rmse, "test_rmse": xgb_test_rmse},
]
spark.createDataFrame(summary).write.mode("overwrite").parquet(f"{OUT_BASE}/run_{run_id}/summary")
print("Saved summary to", f"{OUT_BASE}/run_{run_id}/summary")

In [ ]:
spark.catalog.clearCache()
spark.stop()